In [ ]:
import sys
sys.path.append("..")
from datetime import datetime
import torch, numpy as np

from src.environment.utils import smooth
from src.data import load_and_align_data, PAIRS, get_field

from bokeh.palettes import Category10
import bokeh.plotting as bk
bk.output_notebook()

In [ ]:
PAIRS

# Read Historical Data

In [ ]:
PAIRS_ = {
    'Bitcoin': 'XBTEUR',
    'Ethereum': 'ETHEUR',
    'Ripple': 'XRPEUR',
    'Cardano': 'ADAEUR',
    'Solana': 'SOLEUR',
}

In [ ]:
data, times = load_and_align_data(PAIRS_, interval=1)

In [ ]:
times_ = torch.tensor([t.timestamp() for t in times], dtype=torch.float64)
dt = float(times_.diff().mean().round())
print(f"dt = {dt}")

prices = torch.tensor(get_field(data, 'close')).T
volume = torch.tensor(get_field(data, 'volume')).T

history = [{
    'time'  : t,
    'prices': p,
    'volume': v
} for t, p, v in zip(times_, prices, volume)]

history = sorted(history, key=lambda x: x['time'])

len(history)

In [ ]:
from src.environment.proto_v07 import MultiCurrencyEnv

base_t = 60
tau_p = torch.tensor([base_t*20, base_t*60*3, base_t*60*24, base_t*60*24*7], dtype=torch.float32)
print("tau_p:", (tau_p / (3600 * 24)).tolist(), "[days]")

env = MultiCurrencyEnv(
    N=len(data),
    C0=1_000.0,
    tau_p=tau_p,
    temp=2.0,
    transaction_eps=1e-2,
    bankruptcy_threshold=10.0,
    sell_fee=1.0,
    buy_fee=1.0,
    min_buy_dollars=1.01,
    tax_rate=0.26,
    dV_coeff=1,
)
print(f"state_dim: {env.state_dim} - action_dim: {env.action_dim}")

# Create Agent

In [ ]:
from src.agent.ppo import PPOAgent, RecurrentPPOAgent

agent = RecurrentPPOAgent(
    state_dim=env.state_dim,
    action_dim=env.action_dim,
    hidden_dims=[512, 512],
    hidden_dims_actor=[512],
    hidden_dims_value=[512],
    activation=torch.tanh,
    gamma=0.999,
    eps_clip=0.2,
    gae_lambda=0.95,
    vf_coef=0.5,
    ent_coef=0.01,
    normalize_advantages=False,
    dtype=torch.float32,
    device="cpu"
)
agent.load(f"../data/agent/ppo_v07.ptm")

# Main Training Loop

In [ ]:
loss, rewards, info = [], [], []

In [ ]:
_loss, _rewards, _info = agent.train_on_historical(
    env, history[:int(len(history)*0.9)], n_episodes=10,
    update_interval=512, n_updates=4,
    max_steps=60000, warm_up=12000,
    lr=1e-3, optim="AdamW", init_optimizer=True,
    max_grad_norm=1.0,
)
agent.save(f"../data/agent/ppo_v07.ptm")

loss    += _loss
rewards += _rewards
info    += _info

In [ ]:
# loss_a = torch.tensor([[loss_dict['actor_loss'] for loss_dict in episode_loss] for episode_loss in loss])
t_all = np.concatenate([np.linspace(i,i+1,sum(len(ld["actor_loss"]) for ld in el), endpoint=False) for i, el in enumerate(loss)])
loss_a_all = np.concatenate([np.array([loss_dict['actor_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
# loss_a_mean = np.array([np.mean([loss_dict['actor_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
# loss_a_min = np.array([np.min([loss_dict['actor_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
# loss_a_max = np.array([np.max([loss_dict['actor_loss'] for loss_dict in episode_loss]) for episode_loss in loss])

fig = bk.figure(title="Actor Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320)
# fig.line(torch.arange(loss_a.numel()) / loss_a.shape[1], loss_a.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][0], alpha=0.3)
# fig.line(torch.arange(len(loss_a)) + 0.5, loss_a.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][0])
fig.line(t_all, loss_a_all.reshape(-1), line_width=2, legend_label="Batch Loss", color=Category10[10][0], alpha=0.3)
# fig.line(np.arange(len(loss_a_mean))+0.5, loss_a_mean, line_width=2, legend_label="Epoch Average", color=Category10[10][0])
# fig.varea(np.arange(len(loss_a_mean))+0.5, y1=loss_a_min, y2=loss_a_max, color=Category10[10][0], alpha=0.3)
bk.show(fig)

In [ ]:
# loss_c = torch.tensor([[loss_dict['critic_loss'] for loss_dict in episode_loss] for episode_loss in loss])
loss_c_all = np.concatenate([np.array([loss_dict['critic_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
# loss_c_mean = np.array([np.mean([loss_dict['critic_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
# loss_c_min = np.array([np.min([loss_dict['critic_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
# loss_c_max = np.array([np.max([loss_dict['critic_loss'] for loss_dict in episode_loss]) for episode_loss in loss])

fig = bk.figure(title="Critic Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss [1]", width=900, height=320)
# fig.line(torch.arange(loss_c.numel()) / loss_c.shape[1], loss_c.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][2], alpha=0.3)
# fig.line(torch.arange(len(loss_c)) + 0.5, loss_c.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][2])
fig.line(t_all, loss_c_all.reshape(-1), line_width=2, legend_label="Batch Loss", color=Category10[10][2], alpha=0.3)
# fig.line(np.arange(len(loss_c_mean))+0.5, loss_c_mean, line_width=2, legend_label="Epoch Average", color=Category10[10][2])
# fig.varea(np.arange(len(loss_c_mean))+0.5, y1=loss_c_min, y2=loss_c_max, color=Category10[10][2], alpha=0.3)

bk.show(fig)

In [ ]:
# rewards = torch.tensor(rewards)
rewards_ = [sum(r) for r in rewards]

fig = bk.figure(title="Total Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Loss", width=900, height=320)
# fig.line(torch.arange(rewards.numel()) / rewards.shape[1], rewards.flatten(), line_width=2, legend_label="Reward", color=Category10[10][4], alpha=0.3)
# fig.line(torch.arange(len(rewards)) + 0.5, rewards.mean(1), line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
fig.line(list(range(len(rewards_))), rewards_, line_width=2, legend_label="Total Reward / Episode", color=Category10[10][4])
fig.legend.location = "bottom_right"
bk.show(fig)

In [ ]:
# rewards = torch.tensor(rewards)
rewards_ = [sum(r) / len(r) for r in rewards]
# rewards_min = [min(r) for r in rewards]
# rewards_max = [max(r) for r in rewards]

fig = bk.figure(title="Average Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Reward", width=900, height=320)
# fig.line(torch.arange(rewards.numel()) / rewards.shape[1], rewards.flatten(), line_width=2, legend_label="Reward", color=Category10[10][4], alpha=0.3)
# fig.line(torch.arange(len(rewards)) + 0.5, rewards.mean(1), line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
fig.line(list(range(len(rewards_))), rewards_, line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
# fig.varea(list(range(len(rewards_))), y1=rewards_min, y2=rewards_max, color=Category10[10][4], alpha=0.3)
fig.legend.location = "bottom_right"
bk.show(fig)

In [ ]:
episode = -3

ts = [datetime.fromtimestamp(item['t']) for item in info[episode]]
ps = torch.tensor([item['p'] for item in info[episode]])
Vs = torch.tensor([item['V'] for item in info[episode]])
Cs = torch.tensor([item['C'] for item in info[episode]])
vs = torch.tensor([[w * p  / item['V'] for w, p in zip(item['w'], item['p'])] for item in info[episode]])

f0 = bk.figure(title=f"Episode {episode} - Prices", x_axis_label="t", y_axis_label=r"\(p / p_{max} [1]\)", x_axis_type="datetime", width=900, height=320)
for i, name in enumerate(data):
    f0.line(ts, ps[:,i] / ps[:,i].max(), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
    # f0.line(ts, smooth(ps[:,i] / ps[:,i].max(), [item['t'] for item in info[episode]], tau=60*3), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f0 = bk.figure(title=f"Episode {episode} - Portfolio Fraction", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
f0.line(ts, Cs / Vs, line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(data):
    f0.line(ts, vs[:,i], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f1 = bk.figure(title=f"Episode {episode} - Portfolio Value", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
r1 = f1.line(ts, Vs, line_width=2, legend_label="Total V")
r2 = f1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
f1.legend.click_policy = "hide"
bk.show(f1)

fig = bk.figure(title=f"Episode {episode} - Rewards", x_axis_label="t", y_axis_label=r"Reward \(\left(\log(\frac{V_{t+1}}{V_t})\right)\)", x_axis_type="datetime", width=900, height=320)
fig.scatter(ts, rewards[episode], size=2, color=Category10[10][4], legend_label="Reward")

returns = []
gamma = 0.999
G = 0.0
for r in reversed(rewards[episode]):
    G = r + gamma * G
    returns.insert(0, G)
fig.line(ts, returns, line_width=2, color=Category10[10][5], legend_label="Return")

hist, edges = torch.histogram(torch.tensor(rewards[episode]), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2
figh = bk.figure(title="Reward Distribution", width=300, height=320)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)
fig.legend.click_policy = "hide"

bk.show(bk.row(fig, figh))

In [ ]:
_actions = torch.tensor([item['a'] for item in info[episode]])
ts = [j for j in range(len(_actions))]

f1 = bk.figure(title=f"Action Fraction", x_axis_label="t", y_axis_label="Buy/Sell [USD]", width=900, height=320)
for i in range(env.N):
    f1.scatter(ts, _actions[:,i], size=3, legend_label=f"a_{i+1}", color=Category10[10][(i)%10])
    f1.line(ts, _actions[:,i], line_width=1, line_dash="dashed", legend_label=f"a_{i+1}", color=Category10[10][(i)%10])
f1.legend.click_policy = "hide"
bk.show(f1)

In [ ]:
raise

# Validate

In [ ]:
start = int(len(history)*0.9)

env.save_history = True

hist_s = []
hist_r = []
hist_i = []

state = env.reset(history[start])
for elem in history[start:]:
    a = agent.act(state.to_tensor(), explore=False)
    state, reward, done, _info = env.step(a, data=elem)

    hist_s.append(state)
    hist_r.append(reward)
    hist_i.append(_info)

    if done:
        break

In [ ]:
Vs = [item['V'] for item in hist_i]
Cs = [item['C'] for item in hist_i]
ts = [datetime.fromtimestamp(item["t"]) for item in hist_i]

skip = 10
f1 = bk.figure(
    title=f"Prices",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[start::skip], price[start::skip], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"
bk.show(f1)

fig1 = bk.figure(
    title=f"Portfolio Value",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
fig1.line(ts, Vs, line_width=2, legend_label="Total V")
fig1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
fig1.legend.click_policy = "hide"
bk.show(fig1)

fig2 = bk.figure(
    title=f"Rewards",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Reward (log(V_t+1/V_t))",
)
fig2.line(ts, hist_r, line_width=2, color=Category10[10][4])

hist, edges = torch.histogram(torch.tensor(hist_r), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2

figh = bk.figure(title="Reward Distribution", width=300, height=400)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)

bk.show(bk.row(fig2, figh))

In [ ]:
for i, name in enumerate(data):
    print(i, name)

In [ ]:
skip = 100
Vs = torch.tensor([item["V"] for item in hist_i[::skip]])
Cs = torch.tensor([item["C"] for item in hist_i[::skip]])
ws = torch.stack([item["w"] for item in hist_i[::skip]])
ps = torch.stack([item["p"] for item in hist_i[::skip]])
vs = (ps * ws / Vs[:,None])

print(Vs.shape)
print(Cs.shape)
print(ws.shape)
print(ps.shape)
print(vs.shape)

f = bk.figure(
    title=f"Portfolio Fration",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label=r"\(\text{t}\)",
    y_axis_label=r"\(\text{Fraction} [1]\)",
)
f.line(ts[::skip], Cs / Vs, line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(data):
    f.line(ts[::skip], vs[:,i], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f.legend.click_policy = "hide"
bk.show(f)

skip = 10
f1 = bk.figure(
    title=f"Prices",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label=r"\(\text{t}\)",
    y_axis_label=r"\(p / p_{max} [1]\)",
)
for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[start::skip], price[start::skip] / price.max(), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"
bk.show(f1)